In [ ]:
import numpy as np, random, tensorflow as tf, pandas as pd
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
from skimage.metrics import peak_signal_noise_ratio as psnr, structural_similarity as ssim

SEED = 42
LR = 0.01
NOISE = 0.15

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s)

In [ ]:
(a, b), (c, d) = keras.datasets.mnist.load_data()
X = (np.concatenate([a, c]).astype("float32") / 255.).reshape(-1, 784)
Y = np.concatenate([b, d])

def make_splits(n):
    idx, _ = train_test_split(np.arange(len(X)), train_size=n, stratify=Y, random_state=SEED)
    Xs, Ys = X[idx], Y[idx]
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(Xs, Ys, train_size=0.7, stratify=Ys, random_state=SEED)
    X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=2/3, stratify=y_tmp, random_state=SEED)
    return X_tr, y_tr, X_val, y_val, X_te, y_te

def add_noise(x, level=NOISE, seed=SEED):
    rng = np.random.default_rng(seed)
    return np.clip(x + rng.normal(0, level, x.shape), 0, 1).astype("float32")

In [ ]:
print("train:", a.shape, a.dtype, "| labels:", b.shape)
print("test: ", c.shape, c.dtype, "| labels:", d.shape)
print("pixel range:", a.min(), a.max())
print("total:", len(X), "| classes:", np.unique(Y))

print("train per digit:", np.bincount(b))
print("test per digit: ", np.bincount(d))
print("all 70k per digit:", np.bincount(Y))

In [ ]:
import os, time, json, platform, shutil

MODEL_NAME = "3layer_dense"
LOG_DIR = "logs"


shutil.rmtree(LOG_DIR, ignore_errors=True)
os.makedirs(LOG_DIR, exist_ok=True)

def get_lr(model):
    try:
        return float(keras.backend.get_value(model.optimizer.learning_rate))
    except Exception:
        return float(model.optimizer.learning_rate)

class EpochLogger(keras.callbacks.Callback):
    def __init__(self, size, epochs):
        super().__init__()
        self.size, self.n_epochs = size, epochs
        self.run_id = f"{MODEL_NAME}_n{size}_ep{epochs}_noisy"

    def on_train_begin(self, logs=None):
        self.rows = []
        self.t_start = time.time()

    def on_epoch_begin(self, epoch, logs=None):
        self.t0 = time.time()

    def on_epoch_end(self, epoch, logs=None):
        dt = time.time() - self.t0
        row = dict(
            model=MODEL_NAME, run_id=self.run_id,
            size=self.size, epochs=self.n_epochs, noisy_train=True,
            epoch=epoch + 1,
            loss=logs['loss'], val_loss=logs['val_loss'],
            lr=round(get_lr(self.model), 8),
            epoch_time_s=round(dt, 3),
            elapsed_s=round(time.time() - self.t_start, 2),
        )
        self.rows.append(row)
        print(f"[{self.run_id}] ep {epoch+1:3d}/{self.n_epochs} | "
              f"loss {row['loss']:.5f} | val {row['val_loss']:.5f} | {dt:.2f}s")

    def on_train_end(self, logs=None):
        self.total_time = time.time() - self.t_start
        df = pd.DataFrame(self.rows)
        path = f"{LOG_DIR}/epoch_log.csv"
        df.to_csv(path, mode='a', header=not os.path.exists(path), index=False)
        print(f"Finished in {self.total_time:.1f}s | avg {df.epoch_time_s.mean():.2f}s/epoch\n")

In [ ]:
def build(lr=LR):
    m = keras.Sequential([
        layers.Input((784,)),
        layers.Dense(128, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(784, activation="sigmoid"),
    ])
    m.compile(optimizer=keras.optimizers.Adam(lr), loss="mse")
    return m

def run_all(lr=LR):
    """Denoising: input = noisy image, target = the clean original."""
    out = {}
    for n in [2500, 5000]:
        X_tr, y_tr, X_val, y_val, X_te, y_te = make_splits(n)
        X_in = add_noise(X_tr, seed=SEED+1)
        X_vin = add_noise(X_val, seed=SEED+2)
        X_te_noisy = add_noise(X_te)
        for ep in [50, 100]:
            set_seed()
            model = build(lr)
            logger = EpochLogger(n, ep)
            h = model.fit(X_in, X_tr, validation_data=(X_vin, X_val),
                          epochs=ep, batch_size=64, shuffle=True, verbose=0,
                          callbacks=[logger])
            recon = model.predict(X_te_noisy, verbose=0)
            mse = np.mean((recon - X_te) ** 2)
            out[(n, ep)] = dict(model=model, hist=h.history, recon=recon,
                                noisy=X_te_noisy, clean=X_te, y_te=y_te, mse=mse,
                                time=logger.total_time)
            print(f"n={n} ep={ep} | train={h.history['loss'][-1]:.4f} "
                  f"val={h.history['val_loss'][-1]:.4f} test_mse={mse:.4f}")
    return out

In [ ]:
results = run_all()

In [ ]:
def _finish(fig, save, show):
    if save:
        os.makedirs(os.path.dirname(save), exist_ok=True)
        fig.savefig(save, dpi=150, bbox_inches="tight")
    if show: plt.show()
    else: plt.close(fig)

def plot_loss(res, title="", save=None, show=True):
    fig, axs = plt.subplots(2, 2, figsize=(11, 8))
    for ax, (k, r) in zip(axs.flat, res.items()):
        ax.plot(r['hist']['loss'], label='Training Loss')
        ax.plot(r['hist']['val_loss'], label='Validation Loss')
        ax.set_title(f"{title} n={k[0]}, ep={k[1]}")
        ax.set_xlabel('Epochs'); ax.set_ylabel('Loss'); ax.legend()
    fig.tight_layout()
    _finish(fig, save, show)

def show_digits(res, key, save=None, show=True):
    r = res[key]
    fig, axs = plt.subplots(10, 3, figsize=(4, 14))
    for dgt in range(10):
        i = np.where(r['y_te'] == dgt)[0][0]
        for j, (img, t) in enumerate([(r['clean'][i], 'Original'),
                                      (r['noisy'][i], 'Test'),
                                      (r['recon'][i], 'Result')]):
            axs[dgt, j].imshow(img.reshape(28, 28), cmap='gray')
            axs[dgt, j].axis('off')
            if dgt == 0: axs[dgt, j].set_title(t)
    fig.suptitle(f"size={key[0]}, epochs={key[1]}", y=1.0)
    fig.tight_layout()
    _finish(fig, save, show)

classifiers = {}
def get_clf(n):

    if n not in classifiers:
        X_tr, y_tr, X_val, y_val, *_ = make_splits(n)
        set_seed()
        clf = keras.Sequential([layers.Input((784,)),
                                layers.Dense(128, activation='relu'),
                                layers.Dense(10, activation='softmax')])
        clf.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
        clf.fit(X_tr, y_tr, validation_data=(X_val, y_val),
                epochs=20, batch_size=64, verbose=0)
        classifiers[n] = clf
    return classifiers[n]

def show_confusion(res, key, save=None, show=True):
    r = res[key]; clf = get_clf(key[0])
    fig, axs = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (name, imgs) in zip(axs, [('Original', r['clean']),
                                      ('Noisy', r['noisy']),
                                      ('Denoised', r['recon'])]):
        pred = clf.predict(imgs, verbose=0).argmax(1)
        acc = accuracy_score(r['y_te'], pred)
        ConfusionMatrixDisplay(confusion_matrix(r['y_te'], pred)).plot(ax=ax, cmap='Blues', colorbar=False)
        ax.set_title(f"{name} (acc={acc:.3f})")
    fig.suptitle(f"size={key[0]}, epochs={key[1]}", y=1.02)
    fig.tight_layout()
    _finish(fig, save, show)

def make_table(res):
    rows = []
    for (n, ep), r in res.items():
        clf = get_clf(n)
        c, nz, rc = r['clean'], r['noisy'], r['recon']
        rows.append({
            'size': n, 'epochs': ep,
            'MSE noisy': np.mean((nz - c) ** 2), 'MSE denoised': r['mse'],
            'PSNR noisy': np.mean([psnr(a, b, data_range=1) for a, b in zip(c, nz)]),
            'PSNR denoised': np.mean([psnr(a, b, data_range=1) for a, b in zip(c, rc)]),
            'SSIM noisy': np.mean([ssim(a.reshape(28,28), b.reshape(28,28), data_range=1) for a, b in zip(c, nz)]),
            'SSIM denoised': np.mean([ssim(a.reshape(28,28), b.reshape(28,28), data_range=1) for a, b in zip(c, rc)]),
            'Acc noisy': accuracy_score(r['y_te'], clf.predict(nz, verbose=0).argmax(1)),
            'Acc denoised': accuracy_score(r['y_te'], clf.predict(rc, verbose=0).argmax(1)),
        })
    df = pd.DataFrame(rows)
    f32 = df.select_dtypes("float32").columns
    df[f32] = df[f32].astype("float64")
    return df.round(4)

In [ ]:
def save_results(res):
    df = make_table(res)
    df.insert(0, 'model', MODEL_NAME)
    df['noisy_train'] = True
    df['lr'] = LR
    df['train_time_s'] = [round(r['time'], 1) for r in res.values()]
    path = f"{LOG_DIR}/results_summary.csv"
    df.to_csv(path, mode='a', header=not os.path.exists(path), index=False)
    return df

summary = save_results(results)
display(summary)

config = dict(
    model=MODEL_NAME, task="denoising: noisy input -> clean target",
    seed=SEED, lr=LR, batch_size=64,
    noise_level=NOISE, split="70/20/10 stratified", sizes=[2500, 5000],
    epochs=[50, 100], device="GPU" if tf.config.list_physical_devices("GPU") else "CPU",
    tf_version=tf.__version__, python=platform.python_version(),
)
with open(f"{LOG_DIR}/config.json", "w") as f:
    json.dump(config, f, indent=2)

print(os.listdir(LOG_DIR))
print(pd.read_csv(f"{LOG_DIR}/epoch_log.csv").shape, "rows in epoch_log.csv")

In [ ]:
plot_loss(results, "Noisy→Clean")

In [ ]:
show_digits(results, (5000, 50))

In [ ]:
show_confusion(results, (5000, 50))

In [ ]:
make_table(results)

In [ ]:
m = build()
m.summary()
print(m.input_shape, m.output_shape)

X_tr = make_splits(2500)[0]
out = m.predict(X_tr[:100], verbose=0)
print(out.shape, out.min(), out.max())

In [ ]:
set_seed()
m = build(lr=0.001)
small = X_tr[:64]
h = m.fit(small, small, epochs=500, batch_size=64, verbose=0)
print("start:", h.history['loss'][0], "| end:", h.history['loss'][-1])

In [ ]:
mean_img = X_tr.mean(0)
print("mean-image MSE:", np.mean((X_tr - mean_img) ** 2))

In [ ]:
r = results[(5000, 50)]
print("MSE noisy -> recon :", np.mean((r['recon'] - r['clean'])**2))
print("MSE noisy baseline :", np.mean((r['noisy'] - r['clean'])**2))

In [ ]:

ep_all = pd.read_csv(f"{LOG_DIR}/epoch_log.csv")
os.makedirs(f"{LOG_DIR}/runs", exist_ok=True)
for rid, g in ep_all.groupby("run_id"):
    g.to_csv(f"{LOG_DIR}/runs/{rid}.csv", index=False)
print(sorted(os.listdir(f"{LOG_DIR}/runs")))

In [ ]:

FIG_DIR, TAB_DIR = "figures", "tables"
for d in (FIG_DIR, TAB_DIR):
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d)


plot_loss(results, "Noisy→Clean", save=f"{FIG_DIR}/loss.png", show=False)
for (n, ep) in results:
    show_digits(results, (n, ep), save=f"{FIG_DIR}/digits_n{n}_ep{ep}.png", show=False)
    show_confusion(results, (n, ep), save=f"{FIG_DIR}/confusion_n{n}_ep{ep}.png", show=False)


table = make_table(results)
table["train_time_s"] = [round(r["time"], 1) for r in results.values()]
table.to_csv(f"{TAB_DIR}/results.csv", index=False)

def md_table(df):
    cols = list(df.columns)
    out = ["| " + " | ".join(cols) + " |", "| " + " | ".join("---" for _ in cols) + " |"]
    for r in df.itertuples(index=False):
        out.append("| " + " | ".join(str(v) for v in r) + " |")
    return "\n".join(out)

with open(f"{TAB_DIR}/results.md", "w") as f:
    f.write(md_table(table) + "\n")

cfg = json.load(open(f"{LOG_DIR}/config.json"))
summ = []
next(iter(results.values()))["model"].summary(print_fn=lambda s, **k: summ.append(s))

best = table.loc[table["MSE denoised"].idxmin()]
beats = best["MSE denoised"] < best["MSE noisy"]

fig_lines = "\n".join(
    f"- size {n}, {ep} epochs: [digits](figures/digits_n{n}_ep{ep}.png), "
    f"[confusion matrices](figures/confusion_n{n}_ep{ep}.png)" for (n, ep) in results)

file_rows = [
    ("`*.ipynb`", "the notebook with all code"),
    ("`logs/epoch_log.csv`", "per-epoch loss, validation loss, learning rate and time for every run"),
    ("`logs/results_summary.csv`", "final metrics and training time per run"),
    ("`logs/config.json`", "seed, split, noise, learning rate, runtime and library versions"),
]
if os.path.isdir(f"{LOG_DIR}/runs"):
    file_rows.append(("`logs/runs/*.csv`", "the epoch log split into one file per run"))
file_rows += [
    ("`tables/results.csv`, `tables/results.md`", "the results table"),
    ("`figures/`", "loss curves, digit examples and confusion matrices"),
]
files_md = "\n".join(f"| {a} | {b} |" for a, b in file_rows)

readme = f"""# {cfg['model']}: MNIST image denoising

The model takes a noisy MNIST image and outputs a new image that should match the original. It is trained on pixels only: the target is the clean original image and the loss is mean squared pixel error. This is a denoising / reconstruction task, not classification or prediction. Digit labels are used only to split the data evenly and for the optional accuracy check described under Metrics.

## Setup

| item | value |
| --- | --- |
| data | MNIST, all 70,000 images, stratified subsets of {cfg.get('sizes')} images |
| split | 70% train / 10% validation / 20% test (stratified) |
| noise | Gaussian, sigma = {cfg.get('noise_level')}, clipped to [0, 1]. Added to the input images of train, validation and test (different random noise per split). The targets are always the clean originals. |
| epochs | {cfg.get('epochs')} |
| optimizer | Adam, learning rate {cfg.get('lr')}, batch size {cfg.get('batch_size')} |
| seed | {cfg.get('seed')} (same for every run) |
| runtime | {cfg.get('device')}, TensorFlow {cfg.get('tf_version')}, Python {cfg.get('python')} |

## Model

{chr(10).join(summ).strip()}

## Results

`noisy` columns are the baseline (the noisy test image compared with the clean original). `denoised` columns are the model output compared with the clean original. The model only helps where `denoised` beats `noisy`. MSE: lower is better. PSNR, SSIM, Acc: higher is better.

{md_table(table)}

Lowest denoised MSE: size {int(best['size'])}, {int(best['epochs'])} epochs, MSE {best['MSE denoised']} against a noisy baseline of {best['MSE noisy']} ({'beats' if beats else 'does not beat'} the baseline on MSE).

## Figures

![loss curves](figures/loss.png)

{fig_lines}

## Files

| path | content |
| --- | --- |
{files_md}

## Metrics

- **MSE**: mean squared pixel error against the clean original.
- **PSNR**: peak signal-to-noise ratio in dB, averaged per image.
- **SSIM**: structural similarity, averaged per image.
- **Acc** and the confusion matrices (secondary check): a small dense classifier (784-128-10) trained on clean images is applied to the original, noisy and denoised test images. It shows whether a denoised digit is still read as the right digit. The classifier is not part of the denoiser.

## Reproduce

Open the notebook in Google Colab and run all cells in order. The first logging cell clears `logs/`, so run the whole notebook from the top.
"""
with open("README.md", "w") as f:
    f.write(readme)

print(sorted(os.listdir(FIG_DIR)))
print(os.listdir(TAB_DIR), "| README.md written")